# Bloque 2 — Machine Learning con Spark ML
## Dataset: SIVIGILA — Vigilancia en Salud Pública, Colombia 2019
### Parcial Final — Machine Learning con PySpark y Docker

**Autor:** Sergio Prieto | **Fecha:** Mayo 2026

---
## Objetivo
Aplicar técnicas de machine learning supervisado y no supervisado usando Spark ML:
- **Parte A:** Pipeline de preparación (VectorAssembler, StringIndexer, OneHotEncoder, StandardScaler)
- **Parte B:** PCA + K-Means (reducción de dimensionalidad y clustering)
- **Parte C:** Clasificación supervisada (Regresión Logística + Random Forest)
- **Parte D:** Validación cruzada con CrossValidator

## Variable objetivo
Se agrupan los 69 eventos SIVIGILA en **7 categorías** de salud pública: VECTORIAL, INMUNOPREVENIBLE, MATERNO_INFANTIL, ZOONOTICO, INTOX_VIOLENCIA, CRONICO, OTROS.


In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Fix PySpark Windows
HADOOP_HOME = os.path.join(os.environ.get("TEMP", os.path.expanduser("~")), "hadoop_tmp")
os.environ["HADOOP_HOME"] = HADOOP_HOME
os.makedirs(os.path.join(HADOOP_HOME, "bin"), exist_ok=True)
winutils_path = os.path.join(HADOOP_HOME, "bin", "winutils.exe")
if not os.path.exists(winutils_path):
    with open(winutils_path, "wb") as f: f.write(b"")

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when, regexp_replace, trim, lit, sum as _sum, count,
)
from pyspark.sql.types import IntegerType
from pyspark.ml.feature import (
    StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler, PCA,
)
from pyspark.ml.clustering import KMeans
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml import Pipeline

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"] = 10

CSV_PATH = "../data/sivigila.csv"
OUTPUT_DIR = "../salidas"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Definir categorias (target)
CATEGORIA_MAP = {
    205:"VECTORIAL",210:"VECTORIAL",217:"VECTORIAL",220:"VECTORIAL",
    420:"VECTORIAL",430:"VECTORIAL",440:"VECTORIAL",
    460:"VECTORIAL",470:"VECTORIAL",490:"VECTORIAL",495:"VECTORIAL",
    540:"VECTORIAL",580:"VECTORIAL",895:"VECTORIAL",
    230:"INMUNOPREVENIBLE",320:"INMUNOPREVENIBLE",
    330:"INMUNOPREVENIBLE",340:"INMUNOPREVENIBLE",341:"INMUNOPREVENIBLE",
    345:"INMUNOPREVENIBLE",348:"INMUNOPREVENIBLE",
    500:"INMUNOPREVENIBLE",510:"INMUNOPREVENIBLE",
    520:"INMUNOPREVENIBLE",530:"INMUNOPREVENIBLE",
    620:"INMUNOPREVENIBLE",730:"INMUNOPREVENIBLE",
    760:"INMUNOPREVENIBLE",770:"INMUNOPREVENIBLE",800:"INMUNOPREVENIBLE",
    810:"INMUNOPREVENIBLE",820:"INMUNOPREVENIBLE",825:"INMUNOPREVENIBLE",
    831:"INMUNOPREVENIBLE",
    110:"MATERNO_INFANTIL",112:"MATERNO_INFANTIL",113:"MATERNO_INFANTIL",
    298:"MATERNO_INFANTIL",343:"MATERNO_INFANTIL",
    549:"MATERNO_INFANTIL",550:"MATERNO_INFANTIL",560:"MATERNO_INFANTIL",
    590:"MATERNO_INFANTIL",600:"MATERNO_INFANTIL",
    735:"MATERNO_INFANTIL",740:"MATERNO_INFANTIL",750:"MATERNO_INFANTIL",
    100:"ZOONOTICO",300:"ZOONOTICO",450:"ZOONOTICO",455:"ZOONOTICO",
    228:"INTOX_VIOLENCIA",356:"INTOX_VIOLENCIA",
    360:"INTOX_VIOLENCIA",370:"INTOX_VIOLENCIA",380:"INTOX_VIOLENCIA",
    390:"INTOX_VIOLENCIA",400:"INTOX_VIOLENCIA",410:"INTOX_VIOLENCIA",
    412:"INTOX_VIOLENCIA",414:"INTOX_VIOLENCIA",
    452:"INTOX_VIOLENCIA",875:"INTOX_VIOLENCIA",
    155:"CRONICO",456:"CRONICO",457:"CRONICO",459:"CRONICO",
    850:"CRONICO",305:"OTROS",
}
CATEGORIA_LABEL_MAP = {"VECTORIAL":0,"INMUNOPREVENIBLE":1,"MATERNO_INFANTIL":2,
    "ZOONOTICO":3,"INTOX_VIOLENCIA":4,"CRONICO":5,"OTROS":6}


## Carga y preparación inicial

Se carga el dataset y se mapean las 7 categorías. Se eliminan registros con `COD_DPTO_O = 0` (desconocido).


In [ ]:
spark = SparkSession.builder.appName("SIVIGILA_Bloque2_ML").master("local[*]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.ui.enabled", "false").config("spark.driver.memory", "2g").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

pdf_raw = pd.read_csv(CSV_PATH, dtype=str, encoding="utf-8", keep_default_na=False)
pdf_raw = pdf_raw.rename(columns={"ANO": "ANO_STR"})

cod_to_cat = {str(k): v for k, v in CATEGORIA_MAP.items()}
cod_to_label = {str(k): CATEGORIA_LABEL_MAP[v] for k, v in CATEGORIA_MAP.items()}
pdf_raw["categoria"] = pdf_raw["COD_EVE"].map(cod_to_cat).fillna("OTROS")
pdf_raw["label"] = pdf_raw["COD_EVE"].map(cod_to_label).fillna(6)

df = spark.createDataFrame(pdf_raw)
df = (df
    .withColumn("conteo_casos", col("conteo_casos").cast(IntegerType()))
    .withColumn("SEMANA", col("SEMANA").cast(IntegerType()))
    .withColumn("COD_DPTO_O", col("COD_DPTO_O").cast(IntegerType()))
    .withColumn("COD_MUN_O", col("COD_MUN_O").cast(IntegerType()))
    .withColumn("COD_EVE", col("COD_EVE").cast(IntegerType()))
    .withColumn("label", col("label").cast(IntegerType()))
    .withColumn("ANO", regexp_replace(col("ANO_STR"), r"\.", "").cast(IntegerType()))
    .withColumn("nom_mun", when(trim(col("nom_mun"))=="",None).otherwise(trim(col("nom_mun"))))
    .drop("ANO_STR"))

df = df.withColumn("dpto_str", col("COD_DPTO_O").cast("string"))
df = df.filter(col("COD_DPTO_O") != 0).cache()
total = df.count()
print(f"Registros para ML: {total:,}")

print("\nDistribucion de categorias:")
df.groupBy("categoria","label").count().orderBy("label").show(10, truncate=False)


## Parte A — Pipeline de preparación de datos (Tareas 7-8)

### Tarea 7: Construcción del pipeline
Se construye un pipeline con:
1. **StringIndexer** para `COD_DPTO_O` → índice numérico
2. **OneHotEncoder** (`dropLast=False`) → vector one-hot de 35 posiciones
3. **VectorAssembler** → une `[SEMANA, dpto_ohe, conteo_casos]` en un vector de 37 features
4. **StandardScaler** (`withMean=True, withStd=True`)

### Tarea 8: Justificación de la estandarización
Se eligió **StandardScaler** sobre MinMaxScaler porque:
1. Los algoritmos (LogisticRegression, PCA, K-Means) asumen o se benefician de datos con media 0 y varianza 1
2. Las features tienen escalas muy diferentes: SEMANA [1-52], conteo_casos [1-470], one-hot [0-1]
3. MinMaxScaler es sensible a outliers — el 10% de registros son atípicos en `conteo_casos`


In [ ]:
idx_dpto = StringIndexer(inputCol="dpto_str", outputCol="dpto_idx", handleInvalid="keep")
ohe_dpto = OneHotEncoder(inputCols=["dpto_idx"], outputCols=["dpto_ohe"], dropLast=False)
assembler = VectorAssembler(
    inputCols=["SEMANA", "dpto_ohe", "conteo_casos"],
    outputCol="features", handleInvalid="skip")
scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures",
                        withMean=True, withStd=True)

prep_pipeline = Pipeline(stages=[idx_dpto, ohe_dpto, assembler, scaler])
prep_model = prep_pipeline.fit(df)
df_prep = prep_model.transform(df).cache()
print(f"Registros tras pipeline: {df_prep.count():,}")
print("\nFeatures (5 primeras filas):")
df_prep.select("features","scaledFeatures").show(5, truncate=80)
print(f"\nDimension del vector: {len(df_prep.select('scaledFeatures').first()[0])} features")


## Parte B — Aprendizaje no supervisado (Tareas 9-12)

### Tarea 9: PCA y varianza explicada
Se aplica PCA sobre las 37 features estandarizadas. La varianza se distribuye de forma casi uniforme (~2.9% por componente) debido al peso de las 35 variables one-hot del departamento.

### Tarea 10: Selección de componentes
Se decide conservar K componentes usando dos criterios:
- **≥90% de varianza acumulada**
- **Criterio de Kaiser** (eigenvalue > 1)

### Tarea 11: K-Means + método del codo
Se prueba K de 2 a 10, calculando WCSS/inercia. Se identifica el codo donde la reducción marginal de WCSS cae significativamente.

### Tarea 12: Interpretación de clusters
Se analiza la distribución de categorías reales dentro de cada cluster para caracterizarlos.


In [ ]:
# --- PCA ---
num_features = len(df_prep.select("scaledFeatures").first()[0])
pca = PCA(k=num_features, inputCol="scaledFeatures", outputCol="pcaFeatures")
pca_model = pca.fit(df_prep)
varianza = np.array(pca_model.explainedVariance.toArray())
varianza_acum = np.cumsum(varianza)
var_ratio = varianza_acum / sum(varianza)

print(f"Features totales: {num_features}")
print("\nVarianza explicada (primeras 15 PCs):")
for i in range(min(15, num_features)):
    print(f"  PC{i+1:2d}: {varianza[i]:.4f} ({varianza[i]/sum(varianza)*100:5.1f}%) acum: {var_ratio[i]*100:.1f}%")

# Seleccion de K PCA
k_80 = int(np.argmax(var_ratio >= 0.80)) + 1
k_90 = int(np.argmax(var_ratio >= 0.90)) + 1
K_PCA = max(3, min(k_90, num_features))
print(f"\nK seleccionado: {K_PCA} (explica {var_ratio[K_PCA-1]*100:.1f}%)")

# Grafico PCA
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.bar(range(1, num_features+1), varianza/sum(varianza), color="#2c7bb6")
ax1.axvline(x=K_PCA, color="red", linestyle="--", label=f"K={K_PCA}")
ax1.set_xlabel("Componente"); ax1.set_ylabel("Proporcion de varianza")
ax1.set_title("Scree Plot"); ax1.legend(); ax1.set_xlim(0, min(20, num_features+1))

ax2.plot(range(1, num_features+1), var_ratio, "o-", color="#d7191c", markersize=3)
ax2.axhline(y=0.90, color="gray", linestyle="--", label="90%")
ax2.axvline(x=K_PCA, color="red", linestyle="--", label=f"K={K_PCA}")
ax2.set_xlabel("Componentes"); ax2.set_ylabel("Varianza acumulada")
ax2.set_title("Varianza acumulada"); ax2.legend(); ax2.set_ylim(0,1.05); ax2.set_xlim(0, min(20,num_features+1))
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "fig3_pca_varianza.png"), bbox_inches="tight")
plt.show()


In [ ]:
# Re-entrenar PCA con K optimo
pca_final = PCA(k=K_PCA, inputCol="scaledFeatures", outputCol="pcaFeatures")
pca_model = pca_final.fit(df_prep)
df_pca = pca_model.transform(df_prep).cache()
_ = df_pca.count()

# --- K-Means: metodo del codo ---
print("\nK-Means: metodo del codo")
wcss = []
for k in range(2, 11):
    km = KMeans(k=k, seed=42, featuresCol="pcaFeatures")
    model = km.fit(df_pca)
    wcss.append(model.summary.trainingCost)
    print(f"  K={k}: WCSS = {model.summary.trainingCost:.2f}")

# Elbow plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(2, 11), wcss, "o-", color="#2c7bb6", linewidth=2, markersize=8)
ax.set_xlabel("K"); ax.set_ylabel("WCSS"); ax.set_title("Metodo del codo")
ax.set_xticks(range(2, 11)); ax.grid(True, alpha=0.3)

deltas = [(wcss[i-1]-wcss[i])/wcss[0] for i in range(1, len(wcss))]
k_optimo = 3
for i, d in enumerate(deltas):
    if d < 0.15:
        k_optimo = i + 3
        break
if k_optimo == 3 and deltas:
    k_optimo = int(np.argmax(deltas)) + 3
print(f"\nK optimo: {k_optimo}")
ax.axvline(x=k_optimo, color="red", linestyle="--", label=f"K={k_optimo}")
ax.legend()
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "fig4_elbow_kmeans.png"), bbox_inches="tight")
plt.show()

# K-Means final
km_final = KMeans(k=k_optimo, seed=42, featuresCol="pcaFeatures", predictionCol="cluster")
km_model = km_final.fit(df_pca)
df_clustered = km_model.transform(df_pca).cache()
_ = df_clustered.count()

# Interpretacion
print("\nTamanio de clusters:")
for row in df_clustered.groupBy("cluster").count().orderBy("cluster").collect():
    print(f"  Cluster {row['cluster']}: {row['count']:,} ({row['count']/total*100:.1f}%)")

print("\nCategorias predominantes por cluster:")
for c in range(k_optimo):
    cat_dist = df_clustered.filter(col("cluster")==c).groupBy("categoria").count() \
        .orderBy(col("count").desc()).limit(3).collect()
    total_c = sum(r["count"] for r in cat_dist)
    top = [(r["categoria"], r["count"], r["count"]/total_c*100) for r in cat_dist]
    print(f"  Cluster {c}: " + " | ".join(f"{t[0]}: {t[2]:.0f}%" for t in top))


## Parte C — Clasificación supervisada (Tareas 13-18)

### Tarea 13: Variable objetivo
La variable objetivo es la **categoría del evento** (7 clases). El dataset está relativamente balanceado, con INTOX_VIOLENCIA como clase mayoritaria (~26%) y OTROS como minoritaria (4 registros).

### Tarea 14: Train/test split
División 80/20 con semilla fija (`seed=42`).

### Tarea 15: Modelos entrenados
- **Regresión Logística** multinomial con `maxIter=100`, `regParam=0.1`
- **Random Forest** con `numTrees=50`, `maxDepth=10`

### Tarea 16: Métricas
Se reportan accuracy, precision, recall, F1 y AUC usando `MulticlassClassificationEvaluator` y `sklearn.metrics.roc_auc_score` (one-vs-rest).

### Tarea 17: Matriz de confusión
Se identifican las confusiones más frecuentes entre clases.

### Tarea 18: Importancia de variables
Se extrae la importancia de features del Random Forest y se interpreta.


In [ ]:
# --- Split ---
train, test = df_prep.select("scaledFeatures","label").randomSplit([0.8, 0.2], seed=42)
print(f"Train: {train.count():,} | Test: {test.count():,}")

# --- Modelos ---
lr = LogisticRegression(featuresCol="scaledFeatures", labelCol="label",
                        maxIter=100, regParam=0.1, elasticNetParam=0.0)
lr_model = lr.fit(train)
print("LR entrenada.")

rf = RandomForestClassifier(featuresCol="scaledFeatures", labelCol="label",
                            numTrees=50, maxDepth=10, seed=42)
rf_model = rf.fit(train)
print("RF entrenado.")

# --- Evaluacion ---
evaluators = {
    "Accuracy": MulticlassClassificationEvaluator(labelCol="label",predictionCol="prediction",metricName="accuracy"),
    "F1": MulticlassClassificationEvaluator(labelCol="label",predictionCol="prediction",metricName="f1"),
    "Precision": MulticlassClassificationEvaluator(labelCol="label",predictionCol="prediction",metricName="weightedPrecision"),
    "Recall": MulticlassClassificationEvaluator(labelCol="label",predictionCol="prediction",metricName="weightedRecall"),
}

for name, model in [("Regresion Logistica", lr_model), ("Random Forest", rf_model)]:
    preds = model.transform(test)
    print(f"\n{name}:")
    for m, ev in evaluators.items():
        print(f"  {m}: {ev.evaluate(preds):.4f}")
    # Calcular AUC multiclase (One-vs-Rest) usando sklearn
    pdf_preds = preds.select("label", "prediction", "probability").toPandas()
    from sklearn.metrics import roc_auc_score
    from sklearn.preprocessing import label_binarize
    # Extraer matriz de probabilidades
    proba_cols = [c for c in pdf_preds.columns if c not in ("label", "prediction")]
    y_true_bin = label_binarize(pdf_preds["label"], classes=range(7))
    try:
        # Usar probability array si existe
        if "probability" in pdf_preds.columns:
            proba = np.array(pdf_preds["probability"].apply(lambda v: v.toArray()).tolist())
        else:
            proba = pdf_preds[proba_cols].values
        auc_ovr = roc_auc_score(y_true_bin, proba, multi_class="ovr", average="weighted")
        print(f"  AUC (OvR weighted): {auc_ovr:.4f}")
    except Exception as e:
        print(f"  AUC: no disponible ({e})")


In [ ]:
# --- Matrices de confusion ---
cat_names = {v: k[:12] for k, v in CATEGORIA_LABEL_MAP.items()}
for name, model in [("Regresion Logistica", lr_model), ("Random Forest", rf_model)]:
    preds = model.transform(test)
    confusion = preds.groupBy("label","prediction").count().orderBy("label","prediction").toPandas()
    n_clases = 7
    cm = np.zeros((n_clases, n_clases), dtype=int)
    for _, row in confusion.iterrows():
        cm[int(row["label"])][int(row["prediction"])] = int(row["count"])
    
    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=[cat_names[i] for i in range(n_clases)],
                yticklabels=[cat_names[i] for i in range(n_clases)], ax=ax)
    ax.set_xlabel("Prediccion"); ax.set_ylabel("Real")
    ax.set_title(f"Matriz de Confusion - {name}")
    plt.tight_layout()
    fname = f"fig5_matriz_confusion_{name.replace(' ','_').lower()}.png"
    fig.savefig(os.path.join(OUTPUT_DIR, fname), bbox_inches="tight")
    plt.show()
    
    errores = sorted([(cm[i][j],i,j) for i in range(n_clases) for j in range(n_clases) if i!=j and cm[i][j]>0], reverse=True)
    print(f"\nTop 3 confusiones {name}:")
    for cnt, real, pred in errores[:3]:
        print(f"  Clase {cat_names[real]} predicha como {cat_names[pred]}: {cnt} veces")

# --- Feature importance ---
importances = rf_model.featureImportances.toArray()
fi_data = [("SEMANA", importances[0]), ("conteo_casos", importances[-1]),
           ("Departamento (35 vars)", float(np.sum(importances[1:-1])))]
fi_data.sort(key=lambda x: x[1], reverse=True)
print("\nFeature Importance (RF):")
for feat, imp in fi_data:
    print(f"  {feat}: {imp:.4f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh([f[0] for f in reversed(fi_data)], [f[1] for f in reversed(fi_data)], color="#2c7bb6")
ax.set_xlabel("Importancia"); ax.set_title("Feature Importance - Random Forest")
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "fig6_feature_importance.png"), bbox_inches="tight")
plt.show()


## Parte D — Validación cruzada (Tareas 19-20)

### Tarea 19: CrossValidator
Se aplica validación cruzada con 3 folds sobre Random Forest, explorando 2 hiperparámetros:
- `maxDepth`: [5, 10]
- `numTrees`: [20, 50]

Total: 4 combinaciones × 3 folds = 12 entrenamientos.

### Tarea 20: Modelo ganador
Se reportan los parámetros óptimos y el rendimiento en test del mejor modelo.


In [ ]:
rf_cv = RandomForestClassifier(featuresCol="scaledFeatures", labelCol="label",
                                seed=42, featureSubsetStrategy="sqrt")

param_grid = ParamGridBuilder() \
    .addGrid(rf_cv.maxDepth, [5, 10]) \
    .addGrid(rf_cv.numTrees, [20, 50]) \
    .build()

evaluator_cv = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1")

cv = CrossValidator(estimator=rf_cv, estimatorParamMaps=param_grid,
                    evaluator=evaluator_cv, numFolds=3, seed=42)

print("Ejecutando validacion cruzada (3 folds, 4 combos)...")
cv_model = cv.fit(train)

best_rf = cv_model.bestModel
print(f"\nMejor maxDepth: {best_rf.getOrDefault('maxDepth')}")
print(f"Mejor numTrees: {best_rf.getOrDefault('numTrees')}")
print(f"Mejor F1 (CV avg): {max(cv_model.avgMetrics):.4f}")

best_preds = cv_model.transform(test)
acc = MulticlassClassificationEvaluator(
    labelCol="label",predictionCol="prediction",metricName="accuracy").evaluate(best_preds)
f1 = MulticlassClassificationEvaluator(
    labelCol="label",predictionCol="prediction",metricName="f1").evaluate(best_preds)
print(f"\nMejor modelo sobre test: Accuracy={acc:.4f}, F1={f1:.4f}")

print("\nResultados CV:")
params_list = cv_model.getEstimatorParamMaps()
for i, (params, metric) in enumerate(zip(params_list, cv_model.avgMetrics)):
    d = params[rf_cv.maxDepth]
    n = params[rf_cv.numTrees]
    print(f"  Combo {i+1}: maxDepth={d}, numTrees={n} -> F1 avg = {metric:.4f}")

df.unpersist(); df_prep.unpersist(); df_pca.unpersist(); df_clustered.unpersist()
spark.stop()
print("\nBloque 2 completado.")


---

## CONCLUSIONES DEL BLOQUE 2

### Sobre PCA
La varianza se distribuye de forma casi uniforme entre los 37 componentes debido a que 35 de ellos son variables one-hot del departamento con pesos similares. Se necesitan 32 componentes para explicar el 92.6% de la varianza.

### Sobre K-Means
El método del codo no muestra un codo pronunciado (reducción constante ~3% por cada K adicional), lo cual es esperable dado que las features no tienen una estructura de clusters fuerte. El cluster mayoritario (68.7%) agrupa la mayoría de los datos.

### Sobre clasificación supervisada
- **Accuracy ~29%** (vs 14% aleatorio para 7 clases): los modelos aprenden por encima del azar
- **Precision RF: 41%** — Random Forest es más preciso que Regresión Logística
- El modelo tiende a predecir INTOX_VIOLENCIA (clase mayoritaria) para la mayoría de casos
- **Feature importance:** Departamento (68%) y conteo_casos (31%) son los predictores dominantes; SEMANA contribuye solo 0.27%

### Sobre validación cruzada
El mejor modelo usa `maxDepth=10, numTrees=50` con F1=0.1926. Los árboles más profundos (maxDepth=10) superan a los superficiales (maxDepth=5).

**Limitación:** Las features disponibles (semana, departamento, conteo) no son suficientes para predecir con alta precisión la categoría del evento. Se necesitarían variables clínicas, demográficas y climáticas adicionales.
